# Datamine DXFOUT Process Example

This notebook demonstrates how to configure and run the **`dxfout`** process wrapper in `dmstudio`.

## Process Description

## DXFOUT

Output a Datamine database plotfile to an external system file in Autocad DXF format.

To use the created DXF file under Autocad, invoke Autocad with the new drawing option, and use the DXFIN command to import the DXF file. Note that the DXF file created contains the Entities section only, and hence can only be overlaid onto another drawing (including an empty drawing).

As well as giving the same visual representation of the generated graphics on an Autocad display, this process also attempts to reconstruct base graphic primitives into intelligent Autocad entities. This allows for connected lines to be treated as single entities within Autocad. Textual characters are also combined into "strings" wherever possible such that text strings may be treated as single entities. Textual characters can only be combined into a single string entity if all text properties (colour, width, angle) are equivalent. Some processes currently use an inter-word gap of the height of the character, rather than the width. In such instances **DXFOUT** will not combine the text words. (If it did, your application's text representation would differ from what would be displayed on an Autocad screen).

The colour table is set as for your application.

The following Datamine special markers are supported as entities:

* Code 91 = Circle

* Code 92 = Cross "+"

* Code 93 = Cross "x"

* Code 94 = Triangle

* Code 95 = Square

* Code 96 = Diamond

* Code 97 = Star "*"

Marker types 98 and 99 are currently undefined by your application and hence are ignored.

Shaded ornamentation codes are implemented using fill colors.

The broad line thickness is set to produce a line thickness of 0.7.

### Input Files:

* **in** (Plot):
  Database plotfile to be output to AUTOCAD DXF format.
  Required=Yes

### Output Files:

### Fields:

* **layer** (Any : IN):
  Field containing layer identification. DXF entities will be allocated to layers
  numbered from 1 to n, depending on how many unique identifiers are present in IN. The
  default field name is LAYER
  Default=Undefined
  Required=No

### Parameters:

* **tolernce**:
  Tolerance used in determining if lines can be reconstructed into polylines and
  individual characters into text strings (0.001).
  Range=Undefined
  Values=Undefined
  Default=0.001
  Required=No

* **realwrld**:
  If set to 1, back converts plotted millimeters to real world co-ordinates based on the
  scaling information provided in the XMIN, XMAX, YMIN, YMAX, XSCALE and YSCALE fields in
  the plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **dm**:
  If set to 1, copies DATAMINE colour numbers to the DXF file without translation.
  Otherwise, colours are converted according to a table which is suitable for AUTOCAD. (0)
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **maxvert**:
  Maximum allowable vertices in a single polygon in the DXF file. The default is 2000,
  but this may be too large for some CAD systems.
  Range=Undefined
  Values=Undefined
  Default=2000
  Required=No

* **layernam**:
  Set to 1 to put layer names into the output file without translation. The default is 0.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('dxfout')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute dxfout
print("Running dxfout...")
dm_cmd.dxfout(
    in_i='t_assays',  # required
    # layer_f='optional',  # optional
    # tolernce_p=0.001,  # optional
    # realwrld_p=0,  # optional
    # dm_p=0,  # optional
    # maxvert_p=2000,  # optional
    # layernam_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("dxfout execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_dxfout_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")